# SatQuery AI - retrain: Caption model (caption_v1)

v2 captioner with ImageNet ResNet-50 on RSICD.

**Before running:** Settings -> Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account).

**Run it with "Save Version -> Save & Run All (Commit)"** so it keeps running after the browser closes. Expected time: 1-3 h (estimate; the driver stops training at 10.5 h so the 12 h limit never loses the work).

**Output** (Output tab of the saved version -> Download): `retrained/checkpoints/...` (the weights), `retrained/retrain_manifest_caption.json` (status, datasets, metrics, deviations), `retrained/logs/caption.log`.

Driver and documentation: `training/kaggle/retrain.py`, `docs/kaggle-retrain.md` in https://github.com/hs-zz27/sih2


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import urllib.request
urllib.request.urlopen('https://huggingface.co', timeout=10)
print('internet: OK')

In [ ]:
!rm -rf /kaggle/tmp/sih2 && git clone --depth 1 https://github.com/hs-zz27/sih2.git /kaggle/tmp/sih2
!cd /kaggle/tmp/sih2 && git log --oneline -1

In [ ]:
!cd /kaggle/tmp/sih2 && python training/kaggle/retrain.py --model caption --work /kaggle/tmp/satquery --out /kaggle/working/retrained

In [ ]:
import json, pathlib
m = pathlib.Path('/kaggle/working/retrained/retrain_manifest_caption.json')
if not m.exists():
    raise SystemExit('NO MANIFEST - the driver did not start; read the cell above')
d = json.loads(m.read_text())
print('STATUS:', d['status'])
print('elapsed hours:', d.get('elapsed_hours'), '| GPU:', d.get('gpu'))
for k in ('metrics.json', 'metrics_after_budget.json'):
    if k in d: print(k, json.dumps(d[k])[:400])
!du -sh /kaggle/working/retrained/checkpoints 2>/dev/null
if not d['status'].startswith(('complete', 'stopped by budget')):
    raise SystemExit('RETRAIN DID NOT FINISH: ' + d['status'])